In [1]:
print("Hello, World!")

Hello, World!


In [1]:
#import libraries
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras import Model
from sklearn.model_selection import train_test_split

In [2]:
#import CSV Files For Image data
import pandas as pd
Data = pd.read_csv(r'D:\NITIN_upskill\Agentic_AI\Work\Capstone\archive (2)\images.csv')
Data.head(5)

,image,sender_id,label,kids
0,4285fab0-751a-4b74-8e9b-43af05deee22,124,Not sure,False
1,ea7b6656-3f84-4eb3-9099-23e623fc1018,148,T-Shirt,False
2,00627a3f-0477-401c-95eb-92642cbe078d,94,Not sure,False
3,ea2ffd4d-9b25-4ca8-9dc2-bd27f1cc59fa,43,T-Shirt,False
4,3b86d877-2b9e-4c8b-a6a2-1d87513309d0,189,Shoes,False


In [3]:
#add image path column in Dataframe
Data['image_path'] = Data['image'] + '.jpg'

In [4]:
#check image path coloumn
Data.head(5)

,image,sender_id,label,kids,image_path
0,4285fab0-751a-4b74-8e9b-43af05deee22,124,Not sure,False,4285fab0-751a-4b74-8e9b-43af05deee22.jpg
1,ea7b6656-3f84-4eb3-9099-23e623fc1018,148,T-Shirt,False,ea7b6656-3f84-4eb3-9099-23e623fc1018.jpg
2,00627a3f-0477-401c-95eb-92642cbe078d,94,Not sure,False,00627a3f-0477-401c-95eb-92642cbe078d.jpg
3,ea2ffd4d-9b25-4ca8-9dc2-bd27f1cc59fa,43,T-Shirt,False,ea2ffd4d-9b25-4ca8-9dc2-bd27f1cc59fa.jpg
4,3b86d877-2b9e-4c8b-a6a2-1d87513309d0,189,Shoes,False,3b86d877-2b9e-4c8b-a6a2-1d87513309d0.jpg


In [5]:
#Check isnull values
Data.isnull().sum()

image         0
sender_id     0
label         0
kids          0
image_path    0
dtype: int64

In [6]:
#droup na values from dataframe
Data = Data.dropna()

In [7]:
#check Duplicate values in lables Coloumns
Data.duplicated(subset=['label']).sum()

np.int64(5383)

In [8]:
Data['label'].value_counts()

label
T-Shirt       1011
Longsleeve     699
Pants          692
Shoes          431
Shirt          378
Dress          357
Outwear        312
Shorts         308
Not sure       228
Hat            171
Skirt          155
Polo           120
Undershirt     118
Blazer         109
Hoodie         100
Body            69
Other           67
Top             43
Blouse          23
Skip            12
Name: count, dtype: int64

In [9]:
from sklearn.model_selection import train_test_split

#split the data into training and validation sets
train_data, test_data = train_test_split(
    Data,
    test_size=0.2,
    random_state=42
)

In [10]:
#Split train Data Split into train and validation data
train_data, val_data = train_test_split(train_data, test_size=0.2, random_state=42)

In [11]:
#Data Agmunataion 

IMAGE_SIZE = (224, 224)
Batch_Size = 32

# Augmentation strategy for training data
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

#connect data to data generator
train_generator = train_datagen.flow_from_dataframe(
    train_data,
    directory=r"D:\NITIN_upskill\Agentic_AI\Work\Capstone\archive (2)\images_compressed",
    x_col='image_path',
    y_col='label',
    target_size=IMAGE_SIZE,
    batch_size=Batch_Size,
    class_mode='categorical'
)


Found 3457 validated image filenames belonging to 20 classes.


In [12]:
# Only validation scaling
val_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

val_generator = val_datagen.flow_from_dataframe(
    val_data,
    directory=r"D:\NITIN_upskill\Agentic_AI\Work\Capstone\archive (2)\images_compressed",
    x_col='image_path',
    y_col='label',
    target_size=IMAGE_SIZE,
    batch_size=Batch_Size,
    class_mode='categorical'
)



Found 865 validated image filenames belonging to 20 classes.


In [13]:
# Load pre-trained MobileNetV2 without the top classification layers
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base layers to avoid destroying existing weight representations
base_model.trainable = False

# Construct custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)  # Mitigates overfitting on small subsets
predictions = Dense(len(train_generator.class_indices), activation='softmax')(x)

# Finalize structure
model = Model(inputs=base_model.input, outputs=predictions)

# Compile using Adam optimizer and Categorical Crossentropy loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_cross_entropy',
    metrics=['accuracy']
)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,591,060 (9.88 MB)

 Trainable params: 333,076 (1.27 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [14]:
import os
import tensorflow as tf
from PIL import Image

# Phase 1 Training
# Remove corrupted or invalid image files before training

def valid_image(row):
    path = os.path.join(
        r"D:\NITIN_upskill\Agentic_AI\Work\Capstone\archive (2)\images_compressed",
        row["image_path"]
    )
    try:
        with Image.open(path) as image:
            image.verify()
        return True
    except Exception:
        return False

train_data = train_data[train_data.apply(valid_image, axis=1)].reset_index(drop=True)
val_data = val_data[val_data.apply(valid_image, axis=1)].reset_index(drop=True)

# Recreate generators using only valid images
train_generator = train_datagen.flow_from_dataframe(
    train_data,
    directory=r"D:\NITIN_upskill\Agentic_AI\Work\Capstone\archive (2)\images_compressed",
    x_col="image_path",
    y_col="label",
    classes=list(train_generator.class_indices.keys()),
    target_size=IMAGE_SIZE,
    batch_size=Batch_Size,
    class_mode="categorical",
    shuffle=True
)

val_generator = val_datagen.flow_from_dataframe(
    val_data,
    directory=r"D:\NITIN_upskill\Agentic_AI\Work\Capstone\archive (2)\images_compressed",
    x_col="image_path",
    y_col="label",
    classes=list(train_generator.class_indices.keys()),
    target_size=IMAGE_SIZE,
    batch_size=Batch_Size,
    class_mode="categorical",
    shuffle=False
)

test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255)

test_generator = test_datagen.flow_from_dataframe(
    test_data,  # Your test pandas DataFrame containing test image details
    directory=r"D:\NITIN_upskill\Agentic_AI\Work\Capstone\archive (2)\images_compressed",
    x_col="image_path",
    y_col="label",
    classes=list(train_generator.class_indices.keys()),
    target_size=IMAGE_SIZE,
    batch_size=Batch_Size,
    class_mode="categorical",
    shuffle=False  # Keep shuffle False to correctly pair predictions with true labels
)

Found 3453 validated image filenames belonging to 20 classes.
Found 865 validated image filenames belonging to 20 classes.
Found 1081 validated image filenames belonging to 20 classes.


In [15]:
# Define Early Stopping callbacks for both training phases
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
early_stopping_p1 = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

early_stopping_p2 = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

model_checkpoint_p1 = ModelCheckpoint(
    filepath='fashion_classifier.keras', # Name of the file to save
    monitor='val_loss',                # Watch validation loss
    save_best_only=True                # Only overwrite if performance improves
)

model_checkpoint_p2 = ModelCheckpoint(
    filepath='fashion_classifier.keras', # Name of the file to save
    monitor='val_loss',                # Watch validation loss
    save_best_only=True                # Only overwrite if performance improves
)

In [16]:
# Phase 1 training
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=[early_stopping_p1, model_checkpoint_p1]
)

Epoch 1/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 106s 940ms/step - accuracy: 0.5135 - loss: 1.7217 - val_accuracy: 0.6636 - val_loss: 1.1689
Epoch 2/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 95s 873ms/step - accuracy: 0.6293 - loss: 1.2147 - val_accuracy: 0.7075 - val_loss: 1.0394
Epoch 3/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 88s 815ms/step - accuracy: 0.6609 - loss: 1.0936 - val_accuracy: 0.7156 - val_loss: 0.9746
Epoch 4/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 88s 814ms/step - accuracy: 0.6869 - loss: 0.9955 - val_accuracy: 0.7168 - val_loss: 0.9607
Epoch 5/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 92s 856ms/step - accuracy: 0.7116 - loss: 0.9083 - val_accuracy: 0.7306 - val_loss: 0.9154
Epoch 6/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 86s 799ms/step - accuracy: 0.7237 - loss: 0.8576 - val_accuracy: 0.7272 - val_loss: 0.9232
Epoch 7/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 85s 785ms/step - accuracy: 0.7292 - loss: 0.8433 - val_accuracy: 0.7214 - val_loss: 0.8998
Epoch 8/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 89s 822ms/step - accuracy: 0.7501 - loss: 

In [17]:
# 1. Access the history dictionary
history_dict = history.history

# 2. Extract individual metrics (lists containing values per epoch)
train_loss = history_dict['loss']
val_loss = history_dict['val_loss']
train_acc = history_dict['accuracy']
val_acc = history_dict['val_accuracy']

result={}
#save in Result Dict
result['model_basic']={
    'train_loss' : train_loss,
    'val_loss'  : val_loss,
    'train_acc' : train_acc,
    'val_acc' :val_acc
}

# 3. Print the final epoch's results
print(f"Total Epochs Run: {len(train_loss)}")
print(f"Final Training Accuracy:   {train_acc[-1]:.4f}")
print(f"Final Validation Accuracy: {val_acc[-1]:.4f}")
print(f"Final Training Loss:       {train_loss[-1]:.4f}")
print(f"Final Validation Loss:     {val_loss[-1]:.4f}")


Total Epochs Run: 10
Final Training Accuracy:   0.7625
Final Validation Accuracy: 0.7410
Final Training Loss:       0.7202
Final Validation Loss:     0.8993


In [18]:
# Phase 2: Unfreeze base features for critical alignment
base_model.trainable = True

# Re-compile with an ultra-low learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Final fine tuning adjustment
history_fine = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=[early_stopping_p2, model_checkpoint_p2]
)

Epoch 1/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 389s 3s/step - accuracy: 0.5149 - loss: 1.6309 - val_accuracy: 0.7329 - val_loss: 0.9358
Epoch 2/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 1996s 19s/step - accuracy: 0.6209 - loss: 1.2350 - val_accuracy: 0.7434 - val_loss: 0.9564
Epoch 3/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 302s 3s/step - accuracy: 0.6739 - loss: 1.0678 - val_accuracy: 0.7480 - val_loss: 0.9526
Epoch 4/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 292s 3s/step - accuracy: 0.6884 - loss: 0.9942 - val_accuracy: 0.7445 - val_loss: 0.9510
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [19]:
history_dict_2 = history_fine.history
result['fine_tuned_model'] = {
    'train_loss': history_dict_2['loss'],
    'val_loss': history_dict_2['val_loss'],
    'train_acc': history_dict_2['accuracy'],
    'val_acc': history_dict_2['val_accuracy']
}

print(f"Final Transfer Learning train acc: {history_dict_2['accuracy'][-1]:.4f}")
print(f"Final Transfer Learning val acc:   {history_dict_2['val_accuracy'][-1]:.4f}")

Final Transfer Learning train acc: 0.6884
Final Transfer Learning val acc:   0.7445


In [22]:
#Model  evaluvation
import numpy as np

# 1. Get raw prediction probabilities (Fixed the "test_generator" spelling typo)
test_generator.reset()
# Remove corrupted or missing test images
IMAGE_DIR = r"D:\NITIN_upskill\Agentic_AI\Work\Capstone\archive (2)\images_compressed"

def is_valid_test_image(row):
    path = os.path.join(IMAGE_DIR, row["image_path"])
    try:
        with Image.open(path) as image:
            image.verify()
        return True
    except Exception:
        return False

test_data = test_data[
    test_data.apply(is_valid_test_image, axis=1)
].reset_index(drop=True)

# Recreate the test generator using only valid images
test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

test_generator = test_datagen.flow_from_dataframe(
    test_data,
    directory=IMAGE_DIR,
    x_col="image_path",
    y_col="label",
    classes=list(train_generator.class_indices.keys()),
    target_size=IMAGE_SIZE,
    batch_size=Batch_Size,
    class_mode="categorical",
    shuffle=False
)

y_pred = model.predict(test_generator, verbose=0)

# 2. Get the actual labels and compute test accuracy
test_loss, test_acc = model.evaluate(test_generator, verbose=0)

print(f"\nAccuracy on test set: {test_acc * 100:.2f}%")
print(f"Loss on test set: {test_loss:.4f}")

Found 1080 validated image filenames belonging to 20 classes.

Accuracy on test set: 73.70%
Loss on test set: 0.8573


In [24]:
#Model Comparision
import pandas as pd
Model_Comparison = pd.DataFrame.from_dict(result, orient='index')
Model_Comparison

,train_loss,val_loss,train_acc,val_acc
model_basic,"[1.7217124700546265, 1.2147151231765747, 1.093...","[1.1688919067382812, 1.0394319295883179, 0.974...","[0.5134665369987488, 0.6293078660964966, 0.660...","[0.6635838150978088, 0.7075144648551941, 0.715..."
fine_tuned_model,"[1.6308748722076416, 1.2349778413772583, 1.067...","[0.93583083152771, 0.9564251899719238, 0.95255...","[0.5149145722389221, 0.6209093332290649, 0.673...","[0.7329480051994324, 0.7433525919914246, 0.747..."


In [23]:
import joblib

# Save the trained Keras model
model.save("fashion_classifier.keras")

# Save class mapping and configuration with Joblib
joblib.dump({
    'class_indices': train_generator.class_indices,
    'image_size': IMAGE_SIZE,
    'batch_size': Batch_Size
}, "fashion_classifier_metadata.joblib")

#save the best performing model
from tensorflow.keras.models import load_model
save_model=load_model('fashion_classifier.keras')
save_model.summary()

print("Model and metadata saved successfully.")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 7,704,958 (29.39 MB)

 Trainable params: 2,556,948 (9.75 MB)

 Non-trainable params: 34,112 (133.25 KB)

 Optimizer params: 5,113,898 (19.51 MB)

Model and metadata saved successfully.
